# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khanarmaghanrasheed-18/FlyRankAi-KhanArmaghan-Internship-2026/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

I am following a freestyle lane focused on identifying potential keyword cannibalization and content consolidation opportunities.

The project is primarily an **unsupervised clustering and ranking task**.

Clustering will be used to group pages from the same client that have similar observable characteristics, such as search intent, content type, search performance, position, and engagement. Within each cluster, nearest-neighbor search will identify the pages that are most similar to one another.

The final task is ranking because the useful output is not simply a cluster number. The system will produce a ranked list of page pairs that exhibit characteristics consistent with potential content cannibalization.

The system does not claim that a pair is definitely cannibalizing. It prioritizes candidate pairs for human review.

In [1]:
task_framing = {
    "lane": "Freestyle: content cannibalization risk analysis",
    "learning_type": "Unsupervised learning",
    "methods": [
        "Clustering",
        "Nearest-neighbor similarity search",
        "Candidate-pair ranking"
    ],
    "final_output": "Ranked list of page pairs for human review"
}

task_framing

{'lane': 'Freestyle: content cannibalization risk analysis',
 'learning_type': 'Unsupervised learning',
 'methods': ['Clustering',
  'Nearest-neighbor similarity search',
  'Candidate-pair ranking'],
 'final_output': 'Ranked list of page pairs for human review'}

## Target or proxy

This project does not currently have a supervised target such as `is_cannibalizing = 0 or 1`.

The starter data does not contain verified labels showing which page pairs are truly cannibalizing one another. Therefore, I will not invent such labels and train a classifier as though they were ground truth.

Instead, the unsupervised system will produce two intermediate outputs:

1. a cluster assignment for each page;
2. a similarity score between a page and its nearest pages within the same client and cluster.

The final decision-support output will be a **candidate cannibalization risk score for each page pair**. It may combine page similarity with supporting signals such as shared intent, closeness in average search position, and overlap in visibility.

This score is a ranking score, not a probability and not a confirmed cannibalization label. A high score only means that the pair should be reviewed earlier by an editor.

The eventual output grain will therefore be:

**one row = one candidate pair of pages from the same client.**

In [2]:
import pandas as pd

# This is only a sketch of my future output structure.
# It is not training data and these are not real page values.

candidate_pair_schema = pd.DataFrame({
    "page_a": ["content_A"],
    "page_b": ["content_B"],
    "same_client": [True],
    "similarity_score": [0.91],
    "position_gap": [1.8],
    "candidate_risk_score": [0.84],
    "recommended_action": ["Human consolidation review"]
})

candidate_pair_schema

,page_a,page_b,same_client,similarity_score,position_gap,candidate_risk_score,recommended_action
0,content_A,content_B,True,0.91,1.8,0.84,Human consolidation review


## Success metric

My primary success metric will be **human-reviewed Precision@20**.

Since this project uses an unsupervised approach, there are no ground-truth cannibalization labels available during training. Therefore, traditional supervised metrics such as accuracy, recall, and F1-score are not appropriate.

Instead, I will evaluate the usefulness of the ranked recommendations using human-reviewed Precision@20. The system will generate its top 20 candidate page pairs, and an SEO editor will determine whether each pair is genuinely worth investigating for potential cannibalization or consolidation.

For example, if 15 of the top 20 recommended pairs are considered useful, then:

Precision@20= 15/20 = 0.75

This metric directly measures how useful the recommendation queue is to the editor, which aligns with the project's decision-support objective.

In [3]:
def precision_at_k(reviewed_pairs: pd.DataFrame, k: int = 20) -> float:
    """
    Calculate the proportion of the top-k candidate pairs
    that a human reviewer marked as useful.
    """
    top_k = reviewed_pairs.head(k)

    if len(top_k) == 0:
        return 0.0

    return top_k["useful_for_review"].mean()


# Small illustration of how the future evaluation will work
example_reviews = pd.DataFrame({
    "pair": ["A-B", "C-D", "E-F", "G-H", "I-J"],
    "useful_for_review": [1, 1, 0, 1, 1]
})

example_precision = precision_at_k(example_reviews, k=5)
print(f"Illustrative Precision@5: {example_precision:.2f}")

Illustrative Precision@5: 0.80


## The unit of analysis, as a real dataframe

In the starter dataset, the input unit of analysis is:

**one row = one pseudonymized content page.**

Each page contains observable content and performance information such as its client identifier, search intent, content type, impressions, CTR, average position, word count, and age.

The clustering stage will operate on page-level rows. Pages must only be compared with other pages belonging to the same client because pages from unrelated websites cannot cannibalize one another.

Later, after nearest-neighbor matching, the recommendation output will change to:

**one row = one candidate pair of pages from the same client.**

In [4]:
from pathlib import Path
import pandas as pd

possible_paths = [
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv")
]

data_path = next((path for path in possible_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Could not find data/raw/content_refresh_anonymized.csv. "
        "Run this notebook from inside the cloned internship repository."
    )

df = pd.read_csv(data_path)

page_columns = [
    "content_id",
    "client_id",
    "main_intent",
    "content_type",
    "impressions_90d",
    "ctr",
    "avg_position",
    "word_count",
    "content_age_days"
]

available_page_columns = [
    column for column in page_columns if column in df.columns
]

page_level_data = df[available_page_columns].copy()

print("Dataset shape:", df.shape)
print("Number of page-level rows:", len(page_level_data))
print("Number of clients:", page_level_data["client_id"].nunique())
print("\nOne row represents one pseudonymized content page.")

page_level_data.head()

Dataset shape: (30000, 44)
Number of page-level rows: 30000
Number of clients: 32

One row represents one pseudonymized content page.


,content_id,client_id,main_intent,content_type,impressions_90d,ctr,avg_position,word_count,content_age_days
0,content_304f48230142,client_f369cb89fc,transactional,keyword article,3803,0.76,10.6,3221.0,187
1,content_a1fb4e703a9e,client_4e07408562,informational,keyword article,15320,0.05,20.3,2481.0,445
2,content_9aa793d4d895,client_7f2253d7e2,informational,keyword article,12581,0.09,36.5,3515.0,141
3,content_331d6c4de07b,client_19581e27de,commercial,keyword article,11751,0.49,6.2,NaN,463
4,content_d99b7a2d90ca,client_3fdba35f04,informational,keyword article,19140,0.13,44.0,2803.0,263


In [5]:
# Verify that some clients have multiple pages that can potentially be compared
pages_per_client = (
    page_level_data
    .groupby("client_id")
    .size()
    .sort_values(ascending=False)
)

print("Median pages per client:", pages_per_client.median())
print("Largest number of pages for one client:", pages_per_client.max())

pages_per_client.head()

Median pages per client: 567.0
Largest number of pages for one client: 7008


client_id
client_19581e27de    7008
client_6208ef0f77    3681
client_4e07408562    2294
client_3fdba35f04    2267
client_f369cb89fc    1796
dtype: int64

In [6]:
pair_output_columns = [
    "client_id",
    "page_a",
    "page_b",
    "cluster_id",
    "similarity_score",
    "same_main_intent",
    "content_type_match",
    "position_gap",
    "impressions_gap",
    "ctr_gap",
    "word_count_gap",
    "content_age_gap",
    "candidate_risk_score",
    "review_rank"
]

print("Planned final pair-level output:")
for column in pair_output_columns:
    print("-", column)

Planned final pair-level output:
- client_id
- page_a
- page_b
- cluster_id
- similarity_score
- same_main_intent
- content_type_match
- position_gap
- impressions_gap
- ctr_gap
- word_count_gap
- content_age_gap
- candidate_risk_score
- review_rank


### Pair-level scoring signals beyond cluster similarity

These pair-level features are used to rank candidate page pairs for editor review, not as supervised labels. They help the system highlight pairs that look like strong cannibalization or consolidation opportunities:

- **same_main_intent**: pages targeting the same search intent are more likely to compete.
- **content_type_match**: pages with the same content type are more natural competitors.
- **position_gap**: close average search positions suggest the pages may be competing for the same queries.
- **impressions_gap**: similar visibility indicates the pages are in the same demand band.
- **ctr_gap**: similar click-through rates support the idea that the pages have comparable search performance.
- **word_count_gap**: similar content length may signal overlapping scope.
- **content_age_gap**: age differences help distinguish legacy pages from newer ones and interpret consolidation value.

These signals augment the embedding/cluster-based similarity score and make the candidate risk ranking more informative for human review.


## Why ML beats a fixed rule here

A fixed rule could identify candidates using conditions such as:

- the pages belong to the same client;
- they share the same broad intent;
- both have more than a chosen number of impressions;
- their average positions are close.

However, potential cannibalization is unlikely to be described well by one universal threshold. Page relationships may involve several signals at once, including intent, content type, position, impressions, CTR, engagement, age, and traffic movement. The importance of these signals may also vary across different clients and groups of pages.

An unsupervised method can examine the combined structure of these features without requiring an invented cannibalization label. Clustering can first create smaller groups of pages with similar observable characteristics. Nearest-neighbor search can then identify the closest candidate pages within those groups.

The ML system will not automatically decide that pages should be merged. It will rank pairs that exhibit characteristics consistent with potential cannibalization so editors can review consolidation opportunities.

A fixed rule will still be useful as a transparent baseline. The unsupervised approach should only be considered valuable if its ranked recommendations are more useful under human review than those produced by the simple rule.

In [7]:
baseline_rule = {
    "same_client": True,
    "same_main_intent": True,
    "minimum_impressions": 100,
    "maximum_position_gap": 5
}

ml_approach = {
    "stage_1": "Cluster similar pages within each client",
    "stage_2": "Find nearest neighbors within each cluster",
    "stage_3": "Rank candidate page pairs for human review",
    "comparison": "Compare human-reviewed Precision@20 against the fixed-rule baseline"
}

print("Example fixed-rule baseline:")
print(baseline_rule)

print("\nProposed ML workflow:")
print(ml_approach)

Example fixed-rule baseline:
{'same_client': True, 'same_main_intent': True, 'minimum_impressions': 100, 'maximum_position_gap': 5}

Proposed ML workflow:
{'stage_1': 'Cluster similar pages within each client', 'stage_2': 'Find nearest neighbors within each cluster', 'stage_3': 'Rank candidate page pairs for human review', 'comparison': 'Compare human-reviewed Precision@20 against the fixed-rule baseline'}


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.